In [0]:
campaigns = spark.table('workspace.bronze.campaigns_raw')
customers = spark.table('workspace.bronze.customers_raw')

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

customers.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in customers.columns]).show()

+----------+---------+-----+------------+---------+---+-----------+----+-------+--------+---------+-----------+--------+-----------+-----------+
|first_name|last_name|email|phone_number|user_name|age|customer_id|city|country|latitude|longitude|postal_code|timezone|ingested_at|source_file|
+----------+---------+-----+------------+---------+---+-----------+----+-------+--------+---------+-----------+--------+-----------+-----------+
|         0|        0|    0|           0|        0|  0|          0|   0|      0|       0|        0|          0|       0|          0|          0|
+----------+---------+-----+------------+---------+---+-----------+----+-------+--------+---------+-----------+--------+-----------+-----------+



In [0]:
from pyspark.sql.functions import col, regexp_like

neg_age_count = customers.filter(col('age') < 0).count()
irreal_age_count = customers.filter(col('age') > 120).count()

invalid_emails_df = customers.filter(
    ~col('email').rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,63}$')
)

display(customers)

first_name,last_name,email,phone_number,user_name,age,customer_id,city,country,latitude,longitude,postal_code,timezone,ingested_at,source_file
Anna,Nagy,anna.nagy@yahoo.com,(06)32/181-9600,anagy,23,5c281755-15f8-4bb9-9994-e77a546ee27b,Cegléd,HU,47.3495,19.8347,H-1845,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Péter,Balogh,peter.balogh@gmail.com,38/637 9402,pbalogh,62,695bd2a6-18b5-4c02-a765-c0c27452e9c8,Nagykanizsa,HU,46.6833,17.6356,H-3408,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Richárd,Horváth,richard.horvath@hotmail.com,06-51/161 5594,rhorvath,69,8f503d82-9ec9-4e35-9895-355a6f724329,Esztergom,HU,47.7536,18.7997,H-7096,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
István,Németh,istvan.nemeth@gmail.com,18/495-9310,inemeth,60,1aa3d842-8664-4364-8b76-ee5539b526e8,Kaposvár,HU,46.359,17.7956,H-3533,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Tamás,Horváth,tamas.horvath@gmail.com,06-31/647 5255,thorvath,31,b1928dfb-52c5-4ada-8194-e70b90fd1f66,Veszprém,HU,47.0833,17.9167,H-1109,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Judit,Fodor,judit.fodor@indamail.hu,06-1/928 3276,jfodor,35,ba9f1eb3-ed53-44b0-a643-a1f33366aa6f,Cegléd,HU,47.3495,19.8347,H-4191,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Henrietta,Török,henrietta.torok@citromail.hu,(06)50/305-6413,htorok,76,9a9bfe34-2624-47aa-bf3e-5ee207476017,Miskolc,HU,48.1117,20.7906,H-5009,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Annamária,Bognár,annamaria.bognar@yahoo.com,(06)76/724-2388,abognar,34,8fe6d39b-4a14-459b-86b4-fc3a2b45f5e4,Eger,HU,48.0,20.7833,H-9129,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Eszter,Molnár,eszter.molnar@citromail.hu,53/287-1012,emolnar,58,9c267f14-502f-41e4-ac5f-15fe7f755bb3,Kecskemét,HU,46.8964,19.6897,H-2922,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Gábor,Szűcs,gabor.szucs@citromail.hu,+36 66 978-4801,gszucs,61,835b5302-9f7f-4745-b022-72f1d5c6bf5d,Esztergom,HU,47.7536,18.7997,H-9741,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import re

def fix_email(email):
    if email is None:
        return email
    match = re.match(r'^(.+)@(.+?)(\d*)$', email)
    if match:
        local = match.group(1)
        domain = match.group(2)
        number = match.group(3)
        return f"{local}{number}@{domain}"
    return email

fix_email_udf = udf(fix_email, StringType())

customers = customers.withColumn('email', fix_email_udf(col('email')))

In [0]:
display(customers)


first_name,last_name,email,phone_number,user_name,age,customer_id,city,country,latitude,longitude,postal_code,timezone,ingested_at,source_file
Anna,Nagy,anna.nagy@yahoo.com,(06)32/181-9600,anagy,23,5c281755-15f8-4bb9-9994-e77a546ee27b,Cegléd,HU,47.3495,19.8347,H-1845,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Péter,Balogh,peter.balogh@gmail.com,38/637 9402,pbalogh,62,695bd2a6-18b5-4c02-a765-c0c27452e9c8,Nagykanizsa,HU,46.6833,17.6356,H-3408,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Richárd,Horváth,richard.horvath@hotmail.com,06-51/161 5594,rhorvath,69,8f503d82-9ec9-4e35-9895-355a6f724329,Esztergom,HU,47.7536,18.7997,H-7096,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
István,Németh,istvan.nemeth@gmail.com,18/495-9310,inemeth,60,1aa3d842-8664-4364-8b76-ee5539b526e8,Kaposvár,HU,46.359,17.7956,H-3533,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Tamás,Horváth,tamas.horvath@gmail.com,06-31/647 5255,thorvath,31,b1928dfb-52c5-4ada-8194-e70b90fd1f66,Veszprém,HU,47.0833,17.9167,H-1109,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Judit,Fodor,judit.fodor@indamail.hu,06-1/928 3276,jfodor,35,ba9f1eb3-ed53-44b0-a643-a1f33366aa6f,Cegléd,HU,47.3495,19.8347,H-4191,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Henrietta,Török,henrietta.torok@citromail.hu,(06)50/305-6413,htorok,76,9a9bfe34-2624-47aa-bf3e-5ee207476017,Miskolc,HU,48.1117,20.7906,H-5009,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Annamária,Bognár,annamaria.bognar@yahoo.com,(06)76/724-2388,abognar,34,8fe6d39b-4a14-459b-86b4-fc3a2b45f5e4,Eger,HU,48.0,20.7833,H-9129,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Eszter,Molnár,eszter.molnar@citromail.hu,53/287-1012,emolnar,58,9c267f14-502f-41e4-ac5f-15fe7f755bb3,Kecskemét,HU,46.8964,19.6897,H-2922,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv
Gábor,Szűcs,gabor.szucs@citromail.hu,+36 66 978-4801,gszucs,61,835b5302-9f7f-4745-b022-72f1d5c6bf5d,Esztergom,HU,47.7536,18.7997,H-9741,Europe/Budapest,2026-05-05T22:55:11.173Z,customers.csv


In [0]:
display(campaigns)

campaign_name,currency_code,revenue,channel,event_id,customer_id,clicked,converted,sent_at,opened_at,this_year,unix_time,company_name,business_details,catch_phrase,fake_sentence,product_id,ingested_at,source_file
Multi-channeled fault-tolerant model,LAK,6221.41,SMS,CAMP-2426-JC,0d5d3792-9e67-4a07-96b0-bc8025d59ca7,false,false,2024-07-14T20:16:46.000Z,2025-03-08T20:16:46.000Z,2026,6.46521851366446E8,Lakatos Nagy Zrt.,syndicate efficient web services,Versatile local artificial intelligence,Nulla dolor accusantium eveniet quis quas sequi.,PROD-8953-Jk,2026-05-05T22:55:13.400Z,campaigns.csv
Function-based systemic capability,TTD,43735.82,SMS,CAMP-0575-UB,8a0cf0bf-368c-4c1a-bfe3-0191a2c04ce0,true,false,2024-11-10T09:28:14.000Z,2025-07-18T09:28:14.000Z,2026,2.0757369593371695E8,Horváth és Oláh Kft.,morph ubiquitous networks,Upgradable actuating hierarchy,Esse consectetur dolores facere at.,PROD-6730-Kv,2026-05-05T22:55:13.400Z,campaigns.csv
Ergonomic homogeneous hub,FKP,2232.14,social,CAMP-8309-tv,acd3d26e-86b7-41c5-bc65-6b8982d494f2,true,false,2024-07-22T23:10:44.000Z,2025-01-16T23:10:44.000Z,2026,1.0627364690984789E9,Kovács Lukács Bt.,implement dot-com mindshare,Virtual multi-state initiative,Minus quo rerum ex porro corrupti tempora facilis.,PROD-8228-kT,2026-05-05T22:55:13.400Z,campaigns.csv
Function-based motivating access,XAF,1535.12,social,CAMP-7604-rO,11e78e77-9ba9-4301-b29e-189758d13aaf,true,false,2024-09-05T02:35:22.000Z,2025-01-01T02:35:22.000Z,2026,1.2077703574628413E9,Katona Kft.,mesh killer eyeballs,Fully-configurable holistic database,Ipsa magni exercitationem.,PROD-8467-AO,2026-05-05T22:55:13.400Z,campaigns.csv
Assimilated transitional leverage,XAF,42206.72,SMS,CAMP-7810-xb,e0321335-db67-42da-9b65-afbfd1920dd7,false,false,2024-09-04T06:37:00.000Z,2024-11-08T06:37:00.000Z,2026,3.3012744167861056E8,Balogh és társa Nyrt.,strategize efficient vortals,Reactive directional matrices,Quasi nemo rerum labore corporis.,PROD-3531-Eu,2026-05-05T22:55:13.400Z,campaigns.csv
Triple-buffered user-facing conglomeration,CUP,8750.19,email,CAMP-0705-gI,212b1ee8-16f8-4cf3-9b61-98f5f5de8e56,true,true,2024-09-28T13:17:15.000Z,2025-02-22T13:17:15.000Z,2026,5.297619935034903E8,Kocsis és Szalai Nyrt.,whiteboard granular mindshare,Total regional matrix,Modi itaque eius vitae aperiam.,PROD-4862-BC,2026-05-05T22:55:13.400Z,campaigns.csv
Seamless user-facing adapter,LSL,83193.55,social,CAMP-2958-aQ,cefdcd5b-017d-4ca6-b49e-9f5941842726,false,false,2024-10-14T06:07:59.000Z,2025-07-17T06:07:59.000Z,2026,3.080659693798805E8,Katona és Budai Kkt.,embrace rich e-commerce,Function-based full-range definition,Nostrum explicabo et quisquam quod.,PROD-6419-zC,2026-05-05T22:55:13.400Z,campaigns.csv
Synchronized static approach,BMD,29115.1,SMS,CAMP-4127-cr,d7a3056c-8e73-4877-b343-67d265286d40,false,true,2024-12-04T17:30:14.000Z,2025-11-15T17:30:14.000Z,2026,3.437403990198227E8,Szabó és társa Kkt.,implement scalable networks,Expanded grid-enabled support,Repellendus aperiam beatae quos saepe assumenda qui.,PROD-9722-rj,2026-05-05T22:55:13.400Z,campaigns.csv
Profit-focused incremental conglomeration,ZMW,3242.96,social,CAMP-1579-Sf,40de03a6-c75b-4715-b89b-fb064f3a5b0d,false,false,2024-07-08T01:27:06.000Z,2024-07-20T01:27:06.000Z,2026,1.4406297347515845E9,Papp Katona Kkt.,architect extensible web services,Right-sized composite matrices,Illum aliquid occaecati quisquam ipsa possimus.,PROD-1804-aY,2026-05-05T22:55:13.400Z,campaigns.csv
Customizable background contingency,BWP,10901.09,email,CAMP-2035-Df,e35292e0-6ce8-4e19-a909-6b77ed789ac1,false,false,2025-04-17T19:27:53.000Z,2025-05-04T19:27:53.000Z,2026,1.3341513738861701E9,Tóth Horváth Kft.,optimize open-source e-services,Organic value-added encoding,Recusandae minima id quia aut exercitationem.,PROD-4460-wp,2026-05-05T22:55:13.400Z,campaigns.csv


In [0]:
neg_revenue_count = campaigns.filter(col('revenue') < 0).count()
print(f"Negatív revenue sorok: {neg_revenue_count}")

Negatív revenue sorok: 0


In [0]:
campaigns.select('channel').distinct().show()

from pyspark.sql.functions import lower

campaigns = campaigns.withColumn('channel', lower(col('channel')))

campaigns.select('channel').distinct().show()

+-------+
|channel|
+-------+
|    SMS|
| social|
|  email|
|   push|
+-------+

+-------+
|channel|
+-------+
|    sms|
| social|
|  email|
|   push|
+-------+



In [0]:
silver_customers = customers.select('customer_id', 'city', 'country', 'timezone', 'age')
silver_customers_pii = customers.select('customer_id', 'first_name', 'last_name', 'email', 'phone_number')

silver_customers.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.customers')
silver_customers_pii.write.format('delta').mode('overwrite').saveAsTable('workspace.silver_pii.customers_pii')

print('done')


done


In [0]:
campaigns.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.campaigns')
print('done')

done
